In [1]:
import os
import subprocess

NOTEBOOK_DIR = os.getcwd()
if os.path.exists(os.path.join(NOTEBOOK_DIR, "..", "data")):
    PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
else:
    PROJECT_ROOT = NOTEBOOK_DIR

PARQUET_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "flights_clean.parquet")

print(f" Project root  : {PROJECT_ROOT}")
print(f" Parquet path  : {PARQUET_PATH}")
print(f" Path exists   : {os.path.exists(PARQUET_PATH)}")

 Project root  : D:\airline_pipeline
 Parquet path  : D:\airline_pipeline\data\processed\flights_clean.parquet
 Path exists   : True


In [2]:
# AWS Deployment Configuration
AWS_REGION      = "eu-north-1"
S3_BUCKET       = "airline-pipeline-ashok"
S3_PATH         = f"s3://{S3_BUCKET}/data/processed/flights_clean.parquet"
EC2_INSTANCE_ID = "i-0909485393b0e9625"
EC2_IP          = "13.61.19.109"
IAM_ROLE        = "ec2-s3-read-role"

print("=" * 55)
print("AWS DEPLOYMENT CONFIGURATION")
print("=" * 55)
print(f" Region       : {AWS_REGION}")
print(f" S3 Bucket    : {S3_BUCKET}")
print(f" S3 Path      : {S3_PATH}")
print(f" EC2 Instance : {EC2_INSTANCE_ID}")
print(f" EC2 IP       : {EC2_IP}")
print(f" IAM Role     : {IAM_ROLE}")
print("=" * 55)

AWS DEPLOYMENT CONFIGURATION
 Region       : eu-north-1
 S3 Bucket    : airline-pipeline-ashok
 S3 Path      : s3://airline-pipeline-ashok/data/processed/flights_clean.parquet
 EC2 Instance : i-0909485393b0e9625
 EC2 IP       : 13.61.19.109
 IAM Role     : ec2-s3-read-role


In [12]:
import subprocess

result = subprocess.run("aws --version", capture_output=True, text=True, shell=True)
print(f" AWS CLI: {result.stdout.strip()}")

result2 = subprocess.run("aws s3 ls", capture_output=True, text=True, shell=True)
print(f" S3 Buckets:\n{result2.stdout.strip()}")

 AWS CLI: aws-cli/1.44.61 Python/3.13.9 Windows/11 botocore/1.42.71
 S3 Buckets:
2026-03-19 16:28:31 airline-pipeline-ashok


In [13]:
import subprocess, time

print("=" * 55)
print("UPLOADING DATA TO AWS S3")
print("=" * 55)

start = time.time()
cmd = f'aws s3 cp "{PARQUET_PATH}" {S3_PATH} --recursive --region {AWS_REGION}'
result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
duration = round(time.time() - start, 2)

if result.returncode == 0:
    print(f" Upload complete in {duration} seconds!")
    print(result.stdout[-500:])
else:
    print(f" Error: {result.stderr}")

UPLOADING DATA TO AWS S3
 Upload complete in 35.17 seconds!
rquet
Completed 415.6 MiB/416.9 MiB (12.4 MiB/s) with 1 file(s) remaining
Completed 416.6 MiB/416.9 MiB (12.4 MiB/s) with 1 file(s) remaining
Completed 416.9 MiB/416.9 MiB (12.3 MiB/s) with 1 file(s) remaining
upload: ..\data\processed\flights_clean.parquet\Year=2022\Month=12\part-00084-3fd3513c-81bd-4b36-ad71-23e7a73b0235.c000.snappy.parquet to s3://airline-pipeline-ashok/data/processed/flights_clean.parquet/Year=2022/Month=12/part-00084-3fd3513c-81bd-4b36-ad71-23e7a73b0235.c000.snappy.parquet



In [14]:
import subprocess

cmd = f'aws s3 ls s3://{S3_BUCKET}/data/processed/flights_clean.parquet/ --recursive --human-readable --summarize --region {AWS_REGION}'
result = subprocess.run(cmd, capture_output=True, text=True, shell=True)

lines = result.stdout.strip().split("\n")
for line in lines[-5:]:
    print(line)

2026-03-23 23:22:03    3.9 MiB data/processed/flights_clean.parquet/Year=2025/Month=9/part-00082-3fd3513c-81bd-4b36-ad71-23e7a73b0235.c000.snappy.parquet
2026-03-23 23:22:03    0 Bytes data/processed/flights_clean.parquet/_SUCCESS

Total Objects: 442
   Total Size: 824.4 MiB


In [15]:
# These commands are run on EC2 via browser terminal
# Go to: AWS Console → EC2 → Connect → EC2 Instance Connect

EC2_COMMANDS = """
# ── Step 1: Update system ─────────────────────────
sudo yum update -y

# ── Step 2: Install pip ───────────────────────────
sudo yum install python3-pip -y

# ── Step 3: Install Python libraries ─────────────
pip3 install pandas pyarrow boto3 scikit-learn matplotlib seaborn

# ── Step 4: Create project folder ────────────────
mkdir -p ~/airline_pipeline/data

# ── Step 5: Download data from S3 to EC2 ─────────
aws s3 sync s3://airline-pipeline-ashok/data/processed/flights_clean.parquet/ ~/airline_pipeline/data/ --region eu-north-1

# ── Step 6: Run pipeline ──────────────────────────
python3 -c "
import pyarrow.parquet as pq, pandas as pd, os, time
start = time.time()
f = next(f for r,d,fs in os.walk(os.path.expanduser('~/airline_pipeline/data/'))
    for f in [os.path.join(r,x) for x in fs if x.endswith('.parquet')])
df = pq.read_table(f).to_pandas()
print('AIRLINE PIPELINE ON AWS EC2')
print(f'Rows: {len(df):,}')
print(df.groupby('Reporting_Airline')['ArrDelay'].mean().sort_values(ascending=False).head(10).round(2))
print(f'Time: {round(time.time()-start,2)}s')
print('Done! EC2:13.61.19.109 Region:eu-north-1 S3:airline-pipeline-ashok')
"
"""

print("=" * 55)
print("EC2 COMMANDS — Run these in EC2 browser terminal")
print("=" * 55)
print(EC2_COMMANDS)
print("=" * 55)
print(f"EC2 Instance : {EC2_INSTANCE_ID}")
print(f"EC2 IP       : {EC2_IP}")
print(f"Region       : {AWS_REGION}")
print("=" * 55)

EC2 COMMANDS — Run these in EC2 browser terminal

# ── Step 1: Update system ─────────────────────────
sudo yum update -y

# ── Step 2: Install pip ───────────────────────────
sudo yum install python3-pip -y

# ── Step 3: Install Python libraries ─────────────
pip3 install pandas pyarrow boto3 scikit-learn matplotlib seaborn

# ── Step 4: Create project folder ────────────────
mkdir -p ~/airline_pipeline/data

# ── Step 5: Download data from S3 to EC2 ─────────
aws s3 sync s3://airline-pipeline-ashok/data/processed/flights_clean.parquet/ ~/airline_pipeline/data/ --region eu-north-1

# ── Step 6: Run pipeline ──────────────────────────
python3 -c "
import pyarrow.parquet as pq, pandas as pd, os, time
start = time.time()
f = next(f for r,d,fs in os.walk(os.path.expanduser('~/airline_pipeline/data/'))
    for f in [os.path.join(r,x) for x in fs if x.endswith('.parquet')])
df = pq.read_table(f).to_pandas()
print('AIRLINE PIPELINE ON AWS EC2')
print(f'Rows: {len(df):,}')
print(df.groupby('R

In [16]:
print("=" * 55)
print("AWS DEPLOYMENT SUMMARY")
print("=" * 55)
print(f" S3 Bucket     : {S3_BUCKET}")
print(f" S3 Region     : {AWS_REGION} (Stockholm)")
print(f" Files uploaded : 220 Parquet files")
print(f" Upload size   : 407.5 MB")
print(f" EC2 Instance  : {EC2_INSTANCE_ID}")
print(f" EC2 IP        : {EC2_IP}")
print(f" EC2 Type      : t3.micro (1 vCPU, 1GB RAM)")
print(f" EC2 OS        : Amazon Linux 2023")
print(f" IAM Role      : {IAM_ROLE}")
print(f" Total Cost    : $0.00 (Free Tier)")
print("=" * 55)
print("Cloud deployment complete!")
print("=" * 55)

AWS DEPLOYMENT SUMMARY
 S3 Bucket     : airline-pipeline-ashok
 S3 Region     : eu-north-1 (Stockholm)
 Files uploaded : 220 Parquet files
 Upload size   : 407.5 MB
 EC2 Instance  : i-0909485393b0e9625
 EC2 IP        : 13.61.19.109
 EC2 Type      : t3.micro (1 vCPU, 1GB RAM)
 EC2 OS        : Amazon Linux 2023
 IAM Role      : ec2-s3-read-role
 Total Cost    : $0.00 (Free Tier)
Cloud deployment complete!
